# Week 2: Temporal & Weekday Delay Patterns Analysis
**Focus**: Day-of-week ordering patterns, weekend dispatch backlogs, hourly order distributions, and monthly seasonality.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
df = pd.read_csv('../data/interim/logistics_analytical_dataset.csv')
valid = df[df['is_eligible_otd'] == 1].copy()
print(f"Loaded {len(valid):,} valid delivered orders.")


## 1. Day of Week Fulfillment Analysis
Analyzing whether order placement day affects warehouse dispatch speed and overall delivery duration.


In [ ]:
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_summary = valid.groupby('purchase_weekday').agg(
    order_volume=('order_id', 'count'),
    mean_processing_days=('order_processing_time_days', 'mean'),
    median_processing_days=('order_processing_time_days', 'median'),
    median_transit_days=('carrier_transit_time_days', 'median'),
    median_total_days=('total_delivery_duration_days', 'median'),
    otd_rate_pct=('delivery_outcome', lambda x: (x == 'on_time').mean() * 100),
    delay_rate_pct=('delivery_outcome', lambda x: (x == 'delayed').mean() * 100)
).reindex(days_order)

display(weekday_summary.round(2))


## 2. Weekend Processing Lag Analysis
Evaluating the latency between weekend checkouts and carrier dispatch handoffs.


In [ ]:
valid['is_weekend_order'] = valid['purchase_weekday'].isin(['Saturday', 'Sunday'])
weekend_comp = valid.groupby('is_weekend_order').agg(
    orders=('order_id', 'count'),
    mean_processing_days=('order_processing_time_days', 'mean'),
    median_processing_days=('order_processing_time_days', 'median'),
    otd_rate=('delivery_outcome', lambda x: (x == 'on_time').mean() * 100)
)
weekend_comp.index = ['Weekday Order', 'Weekend Order']
display(weekend_comp.round(2))


## 3. Monthly On-Time Delivery Trajectory (Seasonality & Black Friday Bottleneck)


In [ ]:
monthly_otd = valid[(valid['purchase_year_month'] >= '2017-01') & (valid['purchase_year_month'] <= '2018-08')].groupby('purchase_year_month').agg(
    order_volume=('order_id', 'count'),
    otd_rate=('delivery_outcome', lambda x: (x == 'on_time').mean() * 100),
    delay_rate=('delivery_outcome', lambda x: (x == 'delayed').mean() * 100),
    avg_delivery_duration=('total_delivery_duration_days', 'mean')
)

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

ax1.bar(monthly_otd.index, monthly_otd['order_volume'], color='#c6dbef', alpha=0.7, label='Order Volume')
ax2.plot(monthly_otd.index, monthly_otd['otd_rate'], color='#d62728', marker='o', linewidth=2.5, label='OTD Rate (%)')

ax1.set_xlabel('Purchase Year-Month')
ax1.set_ylabel('Order Volume', color='#2b5c8f')
ax2.set_ylabel('On-Time Delivery Rate (%)', color='#d62728')
ax1.tick_params(axis='x', rotation=45)
plt.title('Monthly Order Volume vs. On-Time Delivery Rate (2017 - 2018)', pad=12, fontweight='bold')
plt.tight_layout()
plt.show()
